# Full-variable models with plain residual scoring

This mirrors TEP notebook 4.4. The lag-2 models use all six variables as predictors for the three endogenous process variables; their residuals are scored by mean squared error without covariance or variance normalization. The six-sample moving-average window and five consecutive alarms are fixed; the threshold comes from faulty validation runs' healthy phases.

In [1]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from IPython.display import display
from carl.causalai_baselines import run_ablation_comparison

ablation_results = run_ablation_comparison("plain_residuals", PROJECT_ROOT)
display(ablation_results["selected_settings"])

,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold
0,plain_residuals,plain,2,6,5,0.793246


# 2. Threshold Selection

Select the threshold as the maximum six-sample moving-average plain residual score in the healthy phases of faulty validation trajectories.

In [2]:
display(ablation_results["validation_metrics"])

,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Detected Runs,Total Faulty Runs,Median Anomaly Detection Delay
0,plain_residuals,plain,2,6,5,0.793246,0.464433,0.337906,0.0,0.505126,0.84,0.68,0.0,0.809524,153,225,33.0


# 3. Test on Faulty Time Series

Evaluate plain residual scoring on every faulty held-out test run.

In [3]:
display(ablation_results["faulty_test_metrics"])
display(ablation_results["faulty_test_by_type"])

,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Detected Runs,Total Faulty Runs,Median Anomaly Detection Delay
0,plain_residuals,plain,2,6,5,0.793246,0.487833,0.366833,0.0,0.536764,0.844444,0.688889,0.0,0.815789,155,225,32.0


,Fault Type,Runs,Detected_Runs,Run_Detection_Rate,Median_Detection_Delay
0,hydraulic_coupling_shift,75,16,0.213333,224.0
1,reaction_coupling_shift,75,75,1.000000,21.0
2,thermal_coupling_shift,75,64,0.853333,41.5


# 4. Test on Fault-Free Time Series

Evaluate the same settings on all healthy held-out test runs.

In [4]:
display(ablation_results["healthy_test_metrics"])
display(ablation_results["comparison"])

,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase FPR,False Alarm Runs,Total Healthy Runs
0,plain_residuals,plain,2,6,5,0.793246,1.0,0.0,0.0,0.0,0.0,0,75


,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold,Faulty Test Sample F1,Faulty Test Trajectory-Phase F1,Faulty Test Trajectory-Phase TPR,Faulty Test Median Detection Delay,Healthy Test Sample FPR,Healthy Test Trajectory-Phase FPR
0,diagonal_covariance,diagonal,2,6,5,3.429457,0.566845,0.891626,0.804444,31.0,0.0,0.0
1,full_var,mahalanobis,2,6,5,3.429900,0.566890,0.891626,0.804444,31.0,0.0,0.0
2,pca,spe,2,6,5,2.360392,0.362259,0.821990,0.697778,52.0,0.0,0.0
4,plain_residuals,plain,2,6,5,0.793246,0.536764,0.815789,0.688889,32.0,0.0,0.0
3,self_history,diagonal,2,6,5,3.495081,0.475545,0.721591,0.564444,32.0,0.0,0.0
